# AfDB Job Data — Exploration Notebook

Scrapes live job listings from the African Development Bank (AfDB) job portal,
normalises them into a canonical 16-field schema, and surfaces a clean DataFrame
ready for ML model selection.

**Completely self-contained — no external repo dependencies.**

---

## How to run on Google Colab

1. Upload this file to [colab.research.google.com](https://colab.research.google.com)
2. Run **Cell 2** first (installs dependencies — takes ~2 min on first run)
3. Then **Runtime → Run all** to execute everything

> **Note:** The scraper opens a headless Chromium browser to interact with the
> SAP Fieldglass portal. Colab supports this natively. Expect ~30–90 seconds
> for the scrape step depending on the number of listings.
> 
> The async Playwright API is used because Colab runs an asyncio event loop
> by default. Top-level `await` in Cell 8 works natively in Colab/IPython.

---

## Configuration

| Option | Default | Description |
|--------|---------|-------------|
| `SEARCH_KEYWORD` | `"data"` | Keyword to search on the AfDB portal |
| `FETCH_DESCRIPTIONS` | `False` | Set `True` to also scrape per-job detail pages (much slower) |

In [ ]:
# Cell 2: Install dependencies
# Run this cell first. Takes ~2-3 minutes on first run.
!pip install -q playwright pandas
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# Cell 3: Imports
# Uses async Playwright API — required because Colab runs an asyncio event loop.
import asyncio
import json
import re
from datetime import datetime, timezone

import pandas as pd
from playwright.async_api import async_playwright, Page

print("Imports OK")

In [ ]:
# Cell 4: Configuration
SEARCH_URL         = "https://afdb1.fcp.eu.fieldglass.cloud.sap/job_search_basic.do"
BASE_URL           = "https://afdb1.fcp.eu.fieldglass.cloud.sap"
SEARCH_KEYWORD     = "data"   # change to "" for all jobs
FETCH_DESCRIPTIONS = False    # set True to scrape per-job detail pages (slow)
RATE_LIMIT_SECONDS = 2
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

In [ ]:
# Cell 5: Scraping helpers (async Playwright API)

async def _extract_rows_from_js(page: Page) -> list[dict]:
    """
    SAP Fieldglass embeds ALL search results in a window-level JS object whose
    name starts with 'jsonObject_search_result_positions_list1_'.
    We ask the browser to return it directly — no pagination needed.
    """
    try:
        raw = await page.evaluate("""
            () => {
                for (const key of Object.keys(window)) {
                    if (key.startsWith('jsonObject_search_result_positions_list1_')) {
                        return JSON.stringify(window[key]);
                    }
                }
                return null;
            }
        """)
    except Exception:
        return []

    if not raw:
        return []

    try:
        obj = json.loads(raw)
    except Exception:
        return []

    results = []
    for row in obj.get("rows", []):
        columns = {col["name"]: col for col in row.get("columns", [])}
        ref_col = columns.get("job_posting_ref", {})
        job_id = ref_col.get("value", "")
        if not job_id:
            continue

        href_match = re.search(r'href\s*[=\\]+\s*["\\/ ]([^"\\>]+)', ref_col.get("html", ""))
        url = ""
        if href_match:
            raw_href = (
                href_match.group(1)
                .replace("\\u003d", "=")
                .replace("\\u0026", "&")
                .replace("\\/", "/")
            )
            url = BASE_URL + "/" + raw_href.lstrip("/") if not raw_href.startswith("http") else raw_href

        results.append({
            "job_id":          job_id,
            "title":           columns.get("title", {}).get("value", ""),
            "location":        columns.get("location", {}).get("value", ""),
            "contract_type":   columns.get("job_type_text", {}).get("value", ""),
            "deadline":        columns.get("end_date", {}).get("value", ""),
            "url":             url,
            "description_raw": "",
        })

    return results


async def _paginate_archivelinks(page: Page) -> list[dict]:
    """Fallback: paginate through DOM a.archiveLink elements page by page."""
    stubs = []
    while True:
        links = await page.locator("a.archiveLink").all()
        for link in links:
            href = await link.get_attribute("href") or ""
            full_url = href if href.startswith("http") else BASE_URL + href
            job_id = (await link.inner_text()).strip()
            if not job_id:
                continue

            title = location = contract_type = deadline = ""
            try:
                row = link.locator("xpath=ancestor::tr").first
                tds = await row.locator("td").all()
                if len(tds) > 1: title         = (await tds[1].inner_text()).strip()
                if len(tds) > 2: location      = (await tds[2].inner_text()).strip()
                if len(tds) > 4: contract_type = (await tds[4].inner_text()).strip()
                if len(tds) > 7: deadline      = (await tds[7].inner_text()).strip()
            except Exception:
                pass

            stubs.append({
                "job_id": job_id, "title": title, "url": full_url,
                "location": location, "contract_type": contract_type,
                "deadline": deadline, "description_raw": "",
            })

        next_btn = page.locator(
            "a[title='Next'], span.pagerNextButton a, "
            "td.pagerNextButton, a:has-text('>'), button:has-text('Next')"
        ).first
        if await next_btn.count() == 0 or not await next_btn.is_visible():
            break
        try:
            await next_btn.click()
            await page.wait_for_selector("a.archiveLink", timeout=15_000)
            await page.wait_for_timeout(1_500)
        except Exception:
            break

    return stubs


async def _scrape_detail(page: Page, stub: dict) -> dict:
    """Navigate to a job detail page and extract the full description."""
    await page.goto(stub["url"], wait_until="load", timeout=60_000)
    await page.wait_for_timeout(3_000)

    async def _text(selector: str) -> str:
        el = page.locator(selector).first
        if await el.count() > 0 and await el.is_visible():
            return (await el.inner_text()).strip()
        return ""

    description_raw = (
        await _text(".jobDescription")
        or await _text(".job-description")
        or await _text("#jobDescription")
        or await _text("div.description")
        or await _text("td.detailsRight")
        or (await page.locator("body").inner_text())[:8000]
    )
    return {**stub, "description_raw": description_raw}


print("Scraping helpers defined")

In [ ]:
# Cell 6: Main scrape function (async)

async def scrape_afdb_jobs(fetch_descriptions: bool = False) -> list[dict]:
    """
    Scrape current AfDB job listings.

    Parameters
    ----------
    fetch_descriptions : bool
        If True, visit each job detail page to get the full description.
        Adds ~3-5 seconds per job. Default False for fast exploration.

    Returns
    -------
    list[dict]
        Raw job dicts with keys: job_id, title, location, contract_type,
        deadline, url, description_raw, scraped_at.
    """
    scraped_at = datetime.now(timezone.utc).isoformat()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()

        try:
            # Load search page
            print(f"Loading: {SEARCH_URL}")
            await page.goto(SEARCH_URL, wait_until="load", timeout=60_000)
            await page.wait_for_timeout(3_000)

            # Fill keyword and submit
            keyword_input = page.locator("input[name='what'], input[type='text']").first
            await keyword_input.wait_for(state="visible", timeout=30_000)
            await keyword_input.fill(SEARCH_KEYWORD)
            await page.locator(
                "input.btn[value='Search'], button.btn:has-text('Search'), input[value='Search']"
            ).first.click()
            print(f"Submitted search for: '{SEARCH_KEYWORD}'")

            # Wait for results
            try:
                await page.wait_for_selector("a.archiveLink", timeout=30_000)
            except Exception:
                print("WARNING: Timed out waiting for results — no listings found.")
                return []
            await page.wait_for_timeout(1_000)

            # Extract listing stubs
            stubs = await _extract_rows_from_js(page)
            if stubs:
                print(f"JS extraction: {len(stubs)} listings found.")
            else:
                print("JS extraction failed — using DOM pagination fallback.")
                stubs = await _paginate_archivelinks(page)
                print(f"DOM pagination: {len(stubs)} listings found.")

            # Optionally fetch descriptions
            if fetch_descriptions:
                print(f"Fetching descriptions for {len(stubs)} jobs (this may take a while)...")
                for i, stub in enumerate(stubs):
                    try:
                        stubs[i] = await _scrape_detail(page, stub)
                        print(f"  [{i+1}/{len(stubs)}] {stubs[i]['job_id']}: {stubs[i]['title'][:60]}")
                    except Exception as exc:
                        print(f"  [{i+1}/{len(stubs)}] WARNING: failed for {stub['job_id']}: {exc}")
                    await asyncio.sleep(RATE_LIMIT_SECONDS)

            return [{**s, "scraped_at": scraped_at} for s in stubs]

        finally:
            await browser.close()


print("scrape_afdb_jobs() defined")

In [ ]:
# Cell 7: Normalise to canonical silver schema

def _extract_country(location: str) -> str:
    """Best-effort: take the last comma-separated part as country."""
    if not location:
        return ""
    parts = [p.strip() for p in location.split(",")]
    return parts[-1] if parts else ""


def normalize_to_silver(raw_jobs: list[dict]) -> list[dict]:
    """
    Map AfDB raw job dicts to the canonical 16-field silver schema used
    across all sources in this project.
    """
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    silver = []
    for job in raw_jobs:
        silver.append({
            "source":           "afdb",
            "source_job_id":    job.get("job_id", ""),
            "title":            job.get("title", ""),
            "organization":     "African Development Bank",
            "location":         job.get("location", ""),
            "country":          _extract_country(job.get("location", "")),
            "remote_flag":      "",
            "contract_type":    job.get("contract_type", ""),
            "grade":            "",
            "posted_at":        "",
            "closes_at":        job.get("deadline", ""),
            "url":              job.get("url", ""),
            "description_text": job.get("description_raw", ""),
            "language":         "en",
            "ingested_at":      job.get("scraped_at", ""),
            "run_id":           run_id,
        })
    return silver


print("normalize_to_silver() defined")

In [ ]:
# Cell 8: Run scraper
# Uses top-level await — supported natively in Colab/IPython.
# Expect 30-90 seconds for the browser to load and scrape.

raw_jobs = await scrape_afdb_jobs(fetch_descriptions=FETCH_DESCRIPTIONS)
print(f"\nScrape complete: {len(raw_jobs)} jobs collected.")

silver = normalize_to_silver(raw_jobs)
print(f"Normalised to {len(silver)} silver records.")

In [ ]:
# Cell 9: Load into DataFrame

df = pd.DataFrame(silver)
print(f"Shape: {df.shape}  ({df.shape[0]} jobs x {df.shape[1]} columns)")
df.head()

In [ ]:
# Cell 10: Missing / empty values
print("=== Empty or missing values per column ===")
empty_counts = (df == "").sum() + df.isnull().sum()
print(empty_counts.to_string())
print()

In [ ]:
# Cell 11: Contract type distribution
print("=== contract_type distribution ===")
print(df["contract_type"].value_counts().to_string())
print()

In [ ]:
# Cell 12: Country / location distribution
print("=== country distribution ===")
print(df["country"].value_counts().to_string())
print()
print("=== location distribution ===")
print(df["location"].value_counts().to_string())

In [ ]:
# Cell 13: Closing dates
print("=== closes_at sample (first 20) ===")
print(df["closes_at"].value_counts().head(20).to_string())

In [ ]:
# Cell 14: Description length
df["desc_length"] = df["description_text"].str.len()
print("=== description_text length stats ===")
print(df["desc_length"].describe().to_string())
if df["desc_length"].max() == 0:
    print()
    print("Tip: re-run with FETCH_DESCRIPTIONS = True to populate description_text.")

---

## Over to you

The `df` DataFrame is ready with all 16 silver fields. From here you can:

- **Set `FETCH_DESCRIPTIONS = True`** in Cell 4 and re-run Cell 8 to populate `description_text` (needed for any text-based model)
- **Save to CSV** with `df.to_csv("afdb_jobs.csv", index=False)`
- **Start model selection** — common approaches for CV-to-job matching:
  - **TF-IDF + cosine similarity** — simple baseline, good interpretability
  - **Sentence embeddings** (e.g. `sentence-transformers`) — better semantic matching
  - **Cross-encoder reranking** — highest quality, slower
  - **LLM scoring** (e.g. Gemini free tier) — the approach already used in afdb-tracker

> **Note on country codes:** AfDB's portal returns ISO 3-letter country codes
> (e.g. `CIV` for Cote d'Ivoire, `TUN` for Tunisia, `MOZ` for Mozambique)
> rather than full country names. This is raw portal data.